# Blood Logistics Decision Support Tool Prototype

**Hackathon: Marine Corps Blood Support Network Risk Modeling**

This notebook demonstrates a decision support tool for blood and blood-product logistics in a Distributed Maritime Operations (DMO) scenario.

### Key Capabilities:
- Hub-and-spoke network risk visualization
- Days of supply remaining & projected stock-out dates
- Route risk assessment
- Market-aware contingency sourcing
- Predictive sustainment modeling

In [ ]:
# Import Libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load and Explore DHA Blood Inventory Data

Load DHA inventory data including blood components, quantities, and location metadata. Explore the data structure and identify key fields for analysis.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define network nodes - Hub and Spokes for DMO scenario
hub = {
    'node_id': 'REGIONAL_HUB',
    'node_name': 'Regional Blood Center (Okinawa)',
    'node_type': 'hub',
    'lat': 26.5,
    'lon': 128.0,
    'capacity_units': 1000,
    'refrigeration_status': 'operational',
    'testing_capacity': 500,
    'lift_availability': 0.85
}

spokes = [
    {'node_id': 'SPOKE_1', 'node_name': 'Marine Expeditionary Force (MEF)', 'node_type': 'spoke', 'lat': 26.5, 'lon': 127.5, 'capacity_units': 200, 'distance_hours': 2},
    {'node_id': 'SPOKE_2', 'node_name': 'Expeditionary Medical Site (EMS) Alpha', 'node_type': 'spoke', 'lat': 15.0, 'lon': 145.0, 'capacity_units': 150, 'distance_hours': 8},
    {'node_id': 'SPOKE_3', 'node_name': 'Naval Amphibious Force', 'node_type': 'spoke', 'lat': 22.0, 'lon': 132.0, 'capacity_units': 180, 'distance_hours': 4},
    {'node_id': 'SPOKE_4', 'node_name': 'Marine Corps Base Hawaii', 'node_type': 'spoke', 'lat': 21.3, 'lon': -157.8, 'capacity_units': 120, 'distance_hours': 12},
    {'node_id': 'SPOKE_5', 'node_name': 'III MEF Support', 'node_type': 'spoke', 'lat': 33.0, 'lon': 131.0, 'capacity_units': 160, 'distance_hours': 6},
    {'node_id': 'SPOKE_6', 'node_name': 'Afloat Element (LHD)', 'node_type': 'spoke', 'lat': 20.0, 'lon': 135.0, 'capacity_units': 100, 'distance_hours': 10},
    {'node_id': 'SPOKE_7', 'node_name': 'Forward Operating Base', 'node_type': 'spoke', 'lat': 10.0, 'lon': 125.0, 'capacity_units': 80, 'distance_hours': 16},
]

nodes = [hub] + spokes
network_df = pd.DataFrame(nodes)

# Blood Product Types
blood_products = [
    {'product_id': 'PRBC', 'product_name': 'Packed Red Blood Cells', 'shelf_days': 42, 'storage_temp': '2-8C'},
    {'product_id': 'PLASM', 'product_name': 'Fresh Frozen Plasma', 'shelf_days': 365, 'storage_temp': '-18C'},
    {'product_id': 'PLT', 'product_name': 'Platelets', 'shelf_days': 5, 'storage_temp': '20-24C'},
    {'product_id': 'CRYO', 'product_name': 'Cryoprecipitate', 'shelf_days': 365, 'storage_temp': '-18C'},
    {'product_id': 'WB', 'product_name': 'Whole Blood', 'shelf_days': 35, 'storage_temp': '2-8C'},
]
products_df = pd.DataFrame(blood_products)

print("Network Nodes:")
display(network_df[['node_id', 'node_name', 'node_type', 'capacity_units']])

print("\nBlood Product Types:")
display(products_df)

In [ ]:
# Generate Inventory Data
def generate_inventory(nodes, products, days_of_data=30):
    inventory = []
    base_date = datetime(2026, 4, 27)
    
    for node in nodes:
        for product in products:
            for day in range(days_of_data):
                current_date = base_date - timedelta(days=days_of_data-day)
                
                # Base inventory varies by node type and product
                base_qty = node['capacity_units'] * np.random.uniform(0.3, 0.8)
                if node['node_type'] == 'hub':
                    base_qty *= 2  # Hub has more stock
                
                # Add expiration tracking
                expiration_days = product['shelf_days']
                days_until_expire = np.random.randint(1, expiration_days)
                expiration_date = current_date + timedelta(days=days_until_expire)
                
                # Daily demand (varies by node)
                daily_demand = int(base_qty * np.random.uniform(0.05, 0.15))
                
                # Current stock
                current_stock = int(base_qty * np.random.uniform(0.2, 1.0))
                
                # Days of supply
                days_supply = current_stock / max(daily_demand, 1)
                
                # Risk factors
                refrigeration_health = np.random.uniform(0.85, 1.0) if node['node_type'] == 'hub' else np.random.uniform(0.7, 1.0)
                route_risk = np.random.uniform(0.1, 0.5) if node['node_type'] == 'spoke' else 0
                
                inventory.append({
                    'date': current_date,
                    'node_id': node['node_id'],
                    'node_name': node['node_name'],
                    'node_type': node['node_type'],
                    'product_id': product['product_id'],
                    'product_name': product['product_name'],
                    'current_stock': current_stock,
                    'daily_demand': daily_demand,
                    'days_supply': round(days_supply, 1),
                    'expiration_date': expiration_date,
                    'days_until_expire': days_until_expire,
                    'refrigeration_health': round(refrigeration_health, 2),
                    'route_risk': round(route_risk, 2),
                    'testing_capacity_available': int(node.get('testing_capacity', 0) * np.random.uniform(0.5, 1.0)),
                })
    
    return pd.DataFrame(inventory)

inventory_df = generate_inventory(nodes, blood_products)
print(f"Generated Inventory Data: {inventory_df.shape}")
display(inventory_df.head(10))

## 2. Build Hub-and-Spoke Network Visualization

Create a network graph representing the regional hub and spoke nodes. Visualize connections between hub and distributed Marine units, afloat elements, and expeditionary medical sites.

In [ ]:
# Network Visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Get hub data
hub_data = network_df[network_df['node_type'] == 'hub'].iloc[0]
spoke_data = network_df[network_df['node_type'] == 'spoke']

# Plot hub
ax.scatter(hub_data['lon'], hub_data['lat'], s=500, c='blue', marker='s', 
           edgecolors='black', linewidth=2, label='Regional Hub', zorder=5)
ax.annotate(f"{hub_data['node_name']}\nCapacity: {hub_data['capacity_units']}", 
            (hub_data['lon'], hub_data['lat']), 
            textcoords="offset points", xytext=(10, -30), fontsize=9, fontweight='bold')

# Plot spokes
for _, spoke in spoke_data.iterrows():
    ax.scatter(spoke['lon'], spoke['lat'], s=200, c='green', marker='o',
               edgecolors='black', linewidth=2, zorder=4)
    ax.annotate(f"{spoke['node_name']}\n{spoke['distance_hours']}h | Cap: {spoke['capacity_units']}", 
                (spoke['lon'], spoke['lat']), 
                textcoords="offset points", xytext=(10, 10), fontsize=8)

# Draw connections from hub to spokes
for _, spoke in spoke_data.iterrows():
    ax.plot([hub_data['lon'], spoke['lon']], [hub_data['lat'], spoke['lat']], 
            'gray', linestyle='--', alpha=0.5, linewidth=1)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Blood Logistics Network - Hub and Spoke Architecture\n(USINDOPACOM DMO Scenario)', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Network visualization complete")

## 3. Analyze Blood Product Expiration Windows

Calculate expiration timelines for blood products. Identify products approaching expiration and model the impact of perishable inventory on sustainment windows.

In [ ]:
# Expiration Analysis
# Get latest inventory data
latest_inventory = inventory_df[inventory_df['date'] == inventory_df['date'].max()]

# Expiration by product
expiration_by_product = latest_inventory.groupby('product_id').agg({
    'days_until_expire': ['min', 'mean', 'max'],
    'current_stock': 'sum'
}).round(1)
expiration_by_product.columns = ['Min Days', 'Avg Days', 'Max Days', 'Total Stock']
expiration_by_product = expiration_by_product.sort_values('Avg Days')

print("Expiration Analysis by Product:")
display(expiration_by_product)

# Visualize expiration windows
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of average days until expiration
ax1 = axes[0]
colors = ['red' if x < 7 else 'orange' if x < 14 else 'green' for x in expiration_by_product['Avg Days']]
expiration_by_product['Avg Days'].plot(kind='bar', ax=ax1, color=colors, edgecolor='black')
ax1.set_title('Average Days Until Expiration by Product', fontweight='bold')
ax1.set_ylabel('Days')
ax1.set_xlabel('Product')
ax1.axhline(y=7, color='red', linestyle='--', label='Critical (7 days)')
ax1.axhline(y=14, color='orange', linestyle='--', label='Warning (14 days)')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Stock with expiration risk
ax2 = axes[1]
expire_soon = latest_inventory[latest_inventory['days_until_expire'] <= 14]
expire_soon_group = expire_soon.groupby('product_id')['current_stock'].sum()
expire_soon_group.plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
ax2.set_title('Stock Expiring Within 14 Days', fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\n⚠️ Products with <7 days shelf life: {len(latest_inventory[latest_inventory['days_until_expire'] < 7])}")
print(f"⚠️ Products with <14 days shelf life: {len(latest_inventory[latest_inventory['days_until_expire'] < 14])}")

## 4. Calculate Days of Supply and Stock-Out Projections

Compute days of supply remaining at each node. Project stock-out dates based on current inventory levels and consumption rates.

In [ ]:
# Stock-Out Prediction
def predict_stockout(inventory_df):
    predictions = []
    
    for node_id in inventory_df['node_id'].unique():
        node_data = inventory_df[inventory_df['node_id'] == node_id]
        
        for product_id in node_data['product_id'].unique():
            product_data = node_data[node_data['product_id'] == product_id].sort_values('date')
            
            if len(product_data) > 0:
                latest = product_data.iloc[-1]
                
                # Calculate projected stock-out date
                if latest['daily_demand'] > 0:
                    days_to_stockout = latest['current_stock'] / latest['daily_demand']
                else:
                    days_to_stockout = 999  # No demand
                
                # Risk category
                if days_to_stockout < 7:
                    risk_category = 'CRITICAL'
                elif days_to_stockout < 14:
                    risk_category = 'WARNING'
                else:
                    risk_category = 'OK'
                
                predictions.append({
                    'node_id': node_id,
                    'node_name': latest['node_name'],
                    'node_type': latest['node_type'],
                    'product_id': product_id,
                    'current_stock': latest['current_stock'],
                    'daily_demand': latest['daily_demand'],
                    'days_to_stockout': round(days_to_stockout, 1),
                    'risk_category': risk_category
                })
    
    return pd.DataFrame(predictions).sort_values('days_to_stockout')

stockout_df = predict_stockout(inventory_df)

print("Stock-Out Predictions (sorted by urgency):")
display(stockout_df.head(15))

# Visualize stock-out timeline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Days to stock-out by node
ax1 = axes[0]
node_stockout = stockout_df.groupby('node_name')['days_to_stockout'].min()
colors = ['red' if x < 7 else 'orange' if x < 14 else 'green' for x in node_stockout.values]
node_stockout.plot(kind='barh', ax=ax1, color=colors, edgecolor='black')
ax1.set_xlabel('Days to Stock-Out')
ax1.set_title('Days to Stock-Out by Node', fontweight='bold')
ax1.axvline(x=7, color='red', linestyle='--', label='Critical (7 days)')
ax1.axvline(x=14, color='orange', linestyle='--', label='Warning (14 days)')
ax1.legend()

# Risk category distribution
ax2 = axes[1]
risk_counts = stockout_df['risk_category'].value_counts()
colors = {'CRITICAL': 'red', 'WARNING': 'orange', 'OK': 'green'}
risk_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', 
                  colors=[colors.get(x, 'gray') for x in risk_counts.index],
                  startangle=90)
ax2.set_title('Risk Category Distribution', fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

print("\n🚨 Critical Stock-Out Risks:")
display(stockout_df[stockout_df['risk_category'] == 'CRITICAL'][['node_name', 'product_id', 'days_to_stockout']])

## 5. Model Casualty Demand Scenarios

Simulate casualty demand signals based on operational scenarios. Model how different casualty assumptions affect inventory depletion across the network.

In [ ]:
# Casualty Demand Scenarios
# Define operational scenarios
scenarios = {
    'PEACETIME': {'casualty_rate': 1.0, 'description': 'Normal operations'},
    'LOW_INTENSITY': {'casualty_rate': 2.5, 'description': 'Limited conflict'},
    'MODERATE': {'casualty_rate': 5.0, 'description': 'Active combat operations'},
    'HIGH_INTENSITY': {'casualty_rate': 10.0, 'description': 'Major conflict scenario'},
}

# Calculate impact of scenarios on stock-out
scenario_impact = []

for scenario_name, scenario_info in scenarios.items():
    multiplier = scenario_info['casualty_rate']
    
    for node_id in inventory_df['node_id'].unique():
        node_data = inventory_df[inventory_df['node_id'] == node_id].iloc[-1]
        
        # Adjusted demand
        adjusted_demand = node_data['daily_demand'] * multiplier
        days_to_stockout = node_data['current_stock'] / max(adjusted_demand, 1)
        
        scenario_impact.append({
            'scenario': scenario_name,
            'description': scenario_info['description'],
            'node': node_data['node_name'],
            'current_stock': node_data['current_stock'],
            'adjusted_daily_demand': adjusted_demand,
            'days_to_stockout': round(days_to_stockout, 1),
            'risk_level': 'CRITICAL' if days_to_stockout < 7 else 'WARNING' if days_to_stockout < 14 else 'OK'
        })

scenario_df = pd.DataFrame(scenario_impact)

print("Casualty Scenario Impact Analysis:")
display(scenario_df.pivot_table(index='node', columns='scenario', values='days_to_stockout').round(1))

# Visualize scenario impact
fig, ax = plt.subplots(figsize=(12, 6))
scenario_pivot = scenario_df.pivot_table(index='node', columns='scenario', values='days_to_stockout')
scenario_pivot.plot(kind='bar', ax=ax, colormap='RdYlGn_r')
ax.set_title('Days to Stock-Out by Operational Scenario', fontweight='bold')
ax.set_ylabel('Days to Stock-Out')
ax.set_xlabel('Node')
ax.axhline(y=7, color='red', linestyle='--', label='Critical Threshold')
ax.axhline(y=14, color='orange', linestyle='--', label='Warning Threshold')
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n✅ Scenario modeling complete - shows how casualty rates affect stock-out timelines")

## 6. Assess Route Risk and Transportation Constraints

Analyze route availability and lift capacity. Identify transportation disruption risks and model the impact of delayed shipments on spoke node viability.

In [ ]:
# Route Risk Assessment
# Calculate route risk based on distance, lift availability, and current conditions

route_risk_data = []

for _, spoke in network_df[network_df['node_type'] == 'spoke'].iterrows():
    # Base route risk from distance
    distance_risk = min(spoke['distance_hours'] / 24, 1.0) * 100  # Normalize to 0-100
    
    # Simulate transportation disruption scenarios
    disruption_scenarios = ['NONE', 'WEATHER', 'HOSTILE', 'LOGISTICS']
    
    for disruption in disruption_scenarios:
        if disruption == 'NONE':
            additional_risk = 0
        elif disruption == 'WEATHER':
            additional_risk = np.random.uniform(10, 25)
        elif disruption == 'HOSTILE':
            additional_risk = np.random.uniform(25, 50)
        else:  # LOGISTICS
            additional_risk = np.random.uniform(15, 35)
        
        total_risk = min(distance_risk + additional_risk, 100)
        
        route_risk_data.append({
            'node': spoke['node_name'],
            'distance_hours': spoke['distance_hours'],
            'disruption_scenario': disruption,
            'route_risk_score': round(total_risk, 1),
            'risk_category': 'CRITICAL' if total_risk > 70 else 'HIGH' if total_risk > 50 else 'MODERATE' if total_risk > 30 else 'LOW'
        })

route_df = pd.DataFrame(route_risk_data)

print("Route Risk Assessment:")
display(route_df.pivot_table(index='node', columns='disruption_scenario', values='route_risk_score').round(1))

# Visualize route risk
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Route risk by node (no disruption)
ax1 = axes[0]
no_disruption = route_df[route_df['disruption_scenario'] == 'NONE'].sort_values('route_risk_score', ascending=False)
colors = ['red' if x > 70 else 'orange' if x > 50 else 'green' for x in no_disruption['route_risk_score']]
ax1.barh(no_disruption['node'], no_disruption['route_risk_score'], color=colors, edgecolor='black')
ax1.set_xlabel('Route Risk Score')
ax1.set_title('Base Route Risk by Node', fontweight='bold')
ax1.axvline(x=50, color='orange', linestyle='--')
ax1.axvline(x=70, color='red', linestyle='--')

# Impact of disruption scenarios
ax2 = axes[1]
avg_risk = route_df.groupby('disruption_scenario')['route_risk_score'].mean()
avg_risk.plot(kind='bar', ax=ax2, color=['green', 'orange', 'red', 'darkred'], edgecolor='black')
ax2.set_title('Average Route Risk by Disruption Scenario', fontweight='bold')
ax2.set_ylabel('Risk Score')
ax2.set_xlabel('Disruption Scenario')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n🚨 High Risk Routes (>50):")
display(route_df[(route_df['route_risk_score'] > 50) & (route_df['disruption_scenario'] == 'NONE')][['node', 'distance_hours', 'route_risk_score']])

## 7. Evaluate Cold-Chain Status and Temperature Excursions

Monitor refrigeration health and cold-chain equipment status. Identify temperature excursion risks that could render blood products unusable.

In [ ]:
# Cold-Chain Status Analysis
# Simulate refrigeration health and temperature excursion risks

cold_chain_data = []

for node in nodes:
    # Refrigeration health (0-1 scale, 1 = perfect)
    refrigeration_health = np.random.uniform(0.7, 1.0) if node['node_type'] == 'spoke' else np.random.uniform(0.85, 1.0)
    
    # Temperature excursion events (simulated)
    temp_excursions = np.random.poisson(2)  # Average 2 excursions per period
    
    # Equipment age factor
    equipment_age = np.random.uniform(0, 5)  # Years
    
    # Calculate cold chain risk score
    health_risk = (1 - refrigeration_health) * 100
    excursion_risk = min(temp_excursions * 15, 100)  # Each excursion adds risk
    age_risk = min(equipment_age * 10, 50)  # Older equipment = more risk
    
    cold_chain_risk = (health_risk * 0.5 + excursion_risk * 0.3 + age_risk * 0.2)
    
    cold_chain_data.append({
        'node_id': node['node_id'],
        'node_name': node['node_name'],
        'node_type': node['node_type'],
        'refrigeration_health': round(refrigeration_health, 2),
        'temp_excursions': temp_excursions,
        'equipment_age_years': round(equipment_age, 1),
        'cold_chain_risk_score': round(cold_chain_risk, 1),
        'risk_category': 'CRITICAL' if cold_chain_risk > 70 else 'WARNING' if cold_chain_risk > 50 else 'OK'
    })

cold_chain_df = pd.DataFrame(cold_chain_data)

print("Cold-Chain Status by Node:")
display(cold_chain_df[['node_name', 'node_type', 'refrigeration_health', 'temp_excursions', 'cold_chain_risk_score', 'risk_category']])

# Visualize cold chain status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cold chain risk by node
ax1 = axes[0]
colors = ['red' if x > 70 else 'orange' if x > 50 else 'green' for x in cold_chain_df['cold_chain_risk_score']]
cold_chain_df.sort_values('cold_chain_risk_score', ascending=False).plot(
    kind='barh', x='node_name', y='cold_chain_risk_score', ax=ax1, 
    color=colors, edgecolor='black', legend=False)
ax1.set_xlabel('Cold Chain Risk Score')
ax1.set_title('Cold Chain Risk by Node', fontweight='bold')
ax1.axvline(x=50, color='orange', linestyle='--', label='Warning')
ax1.axvline(x=70, color='red', linestyle='--', label='Critical')

# Refrigeration health vs risk
ax2 = axes[1]
ax2.scatter(cold_chain_df['refrigeration_health'], cold_chain_df['cold_chain_risk_score'], 
           s=100, c=cold_chain_df['cold_chain_risk_score'], cmap='RdYlGn_r', edgecolors='black')
for _, row in cold_chain_df.iterrows():
    ax2.annotate(row['node_name'][:15], (row['refrigeration_health'], row['cold_chain_risk_score']), 
                fontsize=8, xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('Refrigeration Health')
ax2.set_ylabel('Cold Chain Risk Score')
ax2.set_title('Refrigeration Health vs Risk', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ Nodes with Cold Chain Issues:")
display(cold_chain_df[cold_chain_df['risk_category'] != 'OK'][['node_name', 'temp_excursions', 'cold_chain_risk_score']])

## 8. Identify Secondary Supply Chain Vulnerabilities

Analyze dependencies on testing consumables, transfusion sets, labels, storage containers, and fuel. Model cascading failures from secondary supply shortages.

In [ ]:
# Secondary Supply Chain Vulnerabilities
# Define critical supplies needed for blood support operations

secondary_supplies = [
    {'category': 'Testing', 'item': 'Blood Typing Reagents', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Testing', 'item': 'Crossmatch Reagents', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Testing', 'item': 'Donor Screening Kits', 'criticality': 'CRITICAL', 'days_stock': 14},
    {'category': 'Collection', 'item': 'Blood Collection Bags', 'criticality': 'CRITICAL', 'days_stock': 21},
    {'category': 'Collection', 'item': 'Needles & Syringes', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Transfusion', 'item': 'Transfusion Sets', 'criticality': 'CRITICAL', 'days_stock': 14},
    {'category': 'Transfusion', 'item': 'IV Cannulas', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Storage', 'item': 'Blood Storage Containers', 'criticality': 'HIGH', 'days_stock': 45},
    {'category': 'Storage', 'item': 'Plasma Freezer', 'criticality': 'MEDIUM', 'days_stock': 60},
    {'category': 'Labeling', 'item': 'Blood Product Labels', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Labeling', 'item': 'Barcode Scanners', 'criticality': 'MEDIUM', 'days_stock': 60},
    {'category': 'Power', 'item': 'Generator Fuel', 'criticality': 'CRITICAL', 'days_stock': 7},
    {'category': 'Power', 'item': 'Backup Battery Packs', 'criticality': 'HIGH', 'days_stock': 30},
    {'category': 'Cold Chain', 'item': 'Dry Ice', 'criticality': 'CRITICAL', 'days_stock': 5},
    {'category': 'Cold Chain', 'item': 'Coolant Packs', 'criticality': 'HIGH', 'days_stock': 14},
    {'category': 'Cold Chain', 'item': 'Insulated Shippers', 'criticality': 'HIGH', 'days_stock': 21},
]

supply_df = pd.DataFrame(secondary_supplies)

# Simulate supply levels and vulnerabilities
np.random.seed(42)
supply_levels = []

for _, supply in supply_df.iterrows():
    # Current stock as percentage of max
    stock_pct = np.random.uniform(0.2, 1.0)
    days_remaining = supply['days_stock'] * stock_pct
    
    # Vulnerability score based on criticality and stock level
    criticality_score = {'CRITICAL': 100, 'HIGH': 70, 'MEDIUM': 40}.get(supply['criticality'], 50)
    stock_score = (1 - stock_pct) * 100
    vulnerability = (criticality_score * 0.6 + stock_score * 0.4)
    
    supply_levels.append({
        'category': supply['category'],
        'item': supply['item'],
        'criticality': supply['criticality'],
        'days_stock': supply['days_stock'],
        'current_stock_pct': round(stock_pct * 100, 1),
        'days_remaining': round(days_remaining, 1),
        'vulnerability_score': round(vulnerability, 1),
        'risk_category': 'CRITICAL' if vulnerability > 70 else 'WARNING' if vulnerability > 50 else 'OK'
    })

vuln_df = pd.DataFrame(supply_levels).sort_values('vulnerability_score', ascending=False)

print("Secondary Supply Chain Vulnerabilities:")
display(vuln_df)

# Visualize vulnerabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vulnerability by item
ax1 = axes[0]
colors = ['red' if x == 'CRITICAL' else 'orange' if x == 'HIGH' else 'green' for x in vuln_df['criticality']]
vuln_df.head(10).plot(kind='barh', x='item', y='vulnerability_score', ax=ax1, color=colors[:10], edgecolor='black', legend=False)
ax1.set_xlabel('Vulnerability Score')
ax1.set_title('Top 10 Supply Vulnerabilities', fontweight='bold')
ax1.axvline(x=50, color='orange', linestyle='--')
ax1.axvline(x=70, color='red', linestyle='--')

# By category
ax2 = axes[1]
category_vuln = vuln_df.groupby('category')['vulnerability_score'].mean().sort_values(ascending=False)
category_vuln.plot(kind='bar', ax=ax2, color='coral', edgecolor='black')
ax2.set_title('Average Vulnerability by Category', fontweight='bold')
ax2.set_ylabel('Vulnerability Score')
ax2.set_xlabel('Category')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n🚨 Critical Supply Shortages:")
display(vuln_df[vuln_df['risk_category'] == 'CRITICAL'][['item', 'category', 'days_remaining', 'vulnerability_score']])

## 9. Implement Market-Aware Contingency Sourcing

Evaluate buy-on-market options for approved commercial acquisition. Identify approved vendors for consumables, refrigeration support, dry ice, and transportation services.

In [ ]:
# Market-Aware Contingency Sourcing
# Approved vendors and sourcing options

contingency_supplies = [
    {'category': 'Cold Chain', 'item': 'Portable Refrigeration Unit', 'vendor': 'Helmer Scientific', 'lead_time_days': 3, 'cost_estimate': 15000, 'approval_status': 'APPROVED'},
    {'category': 'Cold Chain', 'item': 'Insulated Shipping Container', 'vendor': 'Pelican BioThermal', 'lead_time_days': 2, 'cost_estimate': 2500, 'approval_status': 'APPROVED'},
    {'category': 'Cold Chain', 'item': 'Dry Ice (500 lb)', 'vendor': 'Local Industrial Gas', 'lead_time_days': 1, 'cost_estimate': 500, 'approval_status': 'APPROVED'},
    {'category': 'Collection', 'item': 'Blood Collection Kits (1000)', 'vendor': 'Haemonetics', 'lead_time_days': 5, 'cost_estimate': 8000, 'approval_status': 'APPROVED'},
    {'category': 'Collection', 'item': 'Donor Screening Supplies', 'vendor': 'Ortho Clinical', 'lead_time_days': 4, 'cost_estimate': 3500, 'approval_status': 'APPROVED'},
    {'category': 'Testing', 'item': 'Blood Typing Reagents', 'vendor': 'Bio-Rad', 'lead_time_days': 3, 'cost_estimate': 2200, 'approval_status': 'APPROVED'},
    {'category': 'Testing', 'item': 'Crossmatch Reagents', 'vendor': 'Immucor', 'lead_time_days': 4, 'cost_estimate': 1800, 'approval_status': 'APPROVED'},
    {'category': 'Transfusion', 'item': 'Transfusion Sets (500)', 'vendor': 'Baxter', 'lead_time_days': 2, 'cost_estimate': 4500, 'approval_status': 'APPROVED'},
    {'category': 'Transfusion', 'item': 'IV Cannulas (200)', 'vendor': 'BD Medical', 'lead_time_days': 2, 'cost_estimate': 1200, 'approval_status': 'APPROVED'},
    {'category': 'Storage', 'item': 'Blood Storage Refrigerator', 'vendor': 'Thermo Fisher', 'lead_time_days': 7, 'cost_estimate': 25000, 'approval_status': 'APPROVED'},
    {'category': 'Power', 'item': 'Generator (10kW)', 'vendor': 'Honda Power', 'lead_time_days': 3, 'cost_estimate': 12000, 'approval_status': 'APPROVED'},
    {'category': 'Power', 'item': 'Fuel (500 gal)', 'vendor': 'Local Fuel Supplier', 'lead_time_days': 1, 'cost_estimate': 2000, 'approval_status': 'HOST NATION'},
    {'category': 'Transport', 'item': 'Validated Shipping Labels', 'vendor': 'Zebra Technologies', 'lead_time_days': 2, 'cost_estimate': 800, 'approval_status': 'APPROVED'},
    {'category': 'Transport', 'item': 'Temperature Monitor', 'vendor': 'Sensitech', 'lead_time_days': 2, 'cost_estimate': 150, 'approval_status': 'APPROVED'},
    {'category': 'Labeling', 'item': 'Blood Product Labels', 'vendor': 'Brady Corporation', 'lead_time_days': 3, 'cost_estimate': 600, 'approval_status': 'APPROVED'},
]

contingency_df = pd.DataFrame(contingency_supplies)

print("Approved Contingency Sourcing Options:")
display(contingency_df)

# Map to vulnerabilities
def recommend_contingency(vuln_df, contingency_df):
    recommendations = []
    
    for _, supply in vuln_df[vuln_df['vulnerability_score'] > 50].iterrows():
        # Find matching contingency items
        matching = contingency_df[contingency_df['category'] == supply['category']]
        
        for _, item in matching.head(2).iterrows():
            recommendations.append({
                'vulnerable_item': supply['item'],
                'category': supply['category'],
                'contingency_item': item['item'],
                'vendor': item['vendor'],
                'lead_time_days': item['lead_time_days'],
                'cost_estimate': item['cost_estimate'],
                'approval_status': item['approval_status']
            })
    
    return pd.DataFrame(recommendations)

recommendations_df = recommend_contingency(vuln_df, contingency_df)

print("\n🎯 Contingency Recommendations for At-Risk Supplies:")
display(recommendations_df)

# Visualize sourcing options
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost by category
ax1 = axes[0]
cost_by_category = contingency_df.groupby('category')['cost_estimate'].sum().sort_values(ascending=False)
cost_by_category.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_title('Estimated Cost by Category', fontweight='bold')
ax1.set_ylabel('Cost ($)')
ax1.set_xlabel('Category')
ax1.tick_params(axis='x', rotation=45)

# Lead time distribution
ax2 = axes[1]
contingency_df['lead_time_days'].hist(bins=7, ax=ax2, color='coral', edgecolor='black')
ax2.set_title('Lead Time Distribution', fontweight='bold')
ax2.set_xlabel('Days')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\n✅ Total contingency options: {len(contingency_df)}")
print(f"✅ Total recommendations: {len(recommendations_df)}")

## 10. Generate Risk Dashboard and Decision Recommendations

Create a summary dashboard showing node-level risk, projected failures, and recommended mitigation actions. Highlight where blood support is most fragile.

In [ ]:
# Calculate Composite Risk Scores
def calculate_composite_risk(inventory_df, cold_chain_df, route_df, vuln_df):
    risk_scores = []
    
    for node in nodes:
        # Inventory risk
        node_inv = inventory_df[inventory_df['node_id'] == node['node_id']]
        avg_days_supply = node_inv['days_supply'].mean()
        inventory_risk = max(0, 100 - (avg_days_supply / 30 * 100))
        
        # Cold chain risk
        node_cc = cold_chain_df[cold_chain_df['node_id'] == node['node_id']]
        cc_risk = node_cc['cold_chain_risk_score'].values[0] if len(node_cc) > 0 else 0
        
        # Route risk (for spokes)
        route_risk_val = route_df[(route_df['node'] == node['node_name']) & 
                                   (route_df['disruption_scenario'] == 'NONE')]['route_risk_score'].values
        route_risk = route_risk_val[0] if len(route_risk_val) > 0 else 0
        
        # Secondary supply risk (average vulnerability)
        supply_risk = vuln_df['vulnerability_score'].mean()
        
        # Composite risk (weighted)
        if node['node_type'] == 'hub':
            composite = (inventory_risk * 0.30 + cc_risk * 0.30 + supply_risk * 0.40)
        else:
            composite = (inventory_risk * 0.25 + cc_risk * 0.20 + route_risk * 0.25 + supply_risk * 0.30)
        
        risk_scores.append({
            'node_id': node['node_id'],
            'node_name': node['node_name'],
            'node_type': node['node_type'],
            'inventory_risk': round(inventory_risk, 1),
            'cold_chain_risk': round(cc_risk, 1),
            'route_risk': round(route_risk, 1),
            'supply_risk': round(supply_risk, 1),
            'composite_risk': round(composite, 1),
            'risk_category': 'CRITICAL' if composite > 70 else 'WARNING' if composite > 50 else 'OK'
        })
    
    return pd.DataFrame(risk_scores).sort_values('composite_risk', ascending=False)

risk_summary_df = calculate_composite_risk(inventory_df, cold_chain_df, route_df, vuln_df)

print("Composite Risk Assessment:")
display(risk_summary_df)

# Create comprehensive dashboard
fig = plt.figure(figsize=(16, 12))

# 1. Composite Risk by Node
ax1 = fig.add_subplot(2, 2, 1)
colors = ['red' if x > 70 else 'orange' if x > 50 else 'green' for x in risk_summary_df['composite_risk']]
bars = ax1.barh(risk_summary_df['node_name'], risk_summary_df['composite_risk'], color=colors, edgecolor='black')
ax1.set_xlabel('Composite Risk Score (0-100)')
ax1.set_title('Node Composite Risk Assessment', fontweight='bold', fontsize=12)
ax1.axvline(x=50, color='orange', linestyle='--', label='Warning')
ax1.axvline(x=70, color='red', linestyle='--', label='Critical')
ax1.legend()

# 2. Risk Component Breakdown
ax2 = fig.add_subplot(2, 2, 2)
risk_components = ['inventory_risk', 'cold_chain_risk', 'route_risk', 'supply_risk']
x = np.arange(len(risk_summary_df))
width = 0.2
for i, comp in enumerate(risk_components):
    ax2.bar(x + i*width, risk_summary_df[comp], width, label=comp.replace('_', ' ').title())
ax2.set_xticks(x + width*1.5)
ax2.set_xticklabels([n[:12] for n in risk_summary_df['node_name']], rotation=45, ha='right')
ax2.set_ylabel('Risk Score')
ax2.set_title('Risk Component Breakdown', fontweight='bold', fontsize=12)
ax2.legend(loc='upper right', fontsize=8)

# 3. Network Map with Risk
ax3 = fig.add_subplot(2, 2, 3)
hub_data = risk_summary_df[risk_summary_df['node_type'] == 'hub'].iloc[0]
spoke_data = risk_summary_df[risk_summary_df['node_type'] == 'spoke']

ax3.scatter(hub_data.get('lon', 128), hub_data.get('lat', 26.5), s=500, c='blue', marker='s', 
           edgecolors='black', linewidth=2, label='Hub', zorder=5)

for _, spoke in network_df[network_df['node_type'] == 'spoke'].iterrows():
    spoke_risk = risk_summary_df[risk_summary_df['node_name'] == spoke['node_name']]['composite_risk'].values
    if len(spoke_risk) > 0:
        color = 'red' if spoke_risk[0] > 70 else 'orange' if spoke_risk[0] > 50 else 'green'
        ax3.scatter(spoke['lon'], spoke['lat'], s=200, c=color, marker='o',
                   edgecolors='black', linewidth=2, zorder=4)
        ax3.annotate(f"{spoke['node_name'][:15]}\n{spoke_risk[0]:.0f}", 
                    (spoke['lon'], spoke['lat']), 
                    textcoords="offset points", xytext=(10, 10), fontsize=8)

# Draw connections
hub_lon, hub_lat = 128, 26.5
for _, spoke in network_df[network_df['node_type'] == 'spoke'].iterrows():
    ax3.plot([hub_lon, spoke['lon']], [hub_lat, spoke['lat']], 'gray', linestyle='--', alpha=0.5)

ax3.set_xlabel('Longitude')
ax3.set_ylabel('Latitude')
ax3.set_title('Network Risk Overlay', fontweight='bold', fontsize=12)
ax3.grid(True, alpha=0.3)

# 4. Risk Category Summary
ax4 = fig.add_subplot(2, 2, 4)
risk_counts = risk_summary_df['risk_category'].value_counts()
colors = {'CRITICAL': 'red', 'WARNING': 'orange', 'OK': 'green'}
risk_counts.plot(kind='pie', ax=ax4, autopct='%1.1f%%', 
                colors=[colors.get(x, 'gray') for x in risk_counts.index],
                startangle=90)
ax4.set_title('Network Risk Status', fontweight='bold', fontsize=12)
ax4.set_ylabel('')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("EXECUTIVE SUMMARY - BLOOD LOGISTICS DECISION SUPPORT")
print("="*80)

print(f"\n📊 Network Overview:")
print(f"   - Total Nodes: {len(nodes)}")
print(f"   - Hub: 1 (Regional Blood Center - Okinawa)")
print(f"   - Spokes: {len(spokes)}")
print(f"   - Blood Products Tracked: {len(blood_products)}")

critical_nodes = risk_summary_df[risk_summary_df['risk_category'] == 'CRITICAL']
warning_nodes = risk_summary_df[risk_summary_df['risk_category'] == 'WARNING']

print(f"\n🚨 Risk Status:")
print(f"   - CRITICAL Risk Nodes: {len(critical_nodes)}")
print(f"   - WARNING Risk Nodes: {len(warning_nodes)}")
print(f"   - OK Risk Nodes: {len(risk_summary_df) - len(critical_nodes) - len(warning_nodes)}")

critical_stockouts = stockout_df[stockout_df['risk_category'] == 'CRITICAL']
print(f"\n⚠️ Stock-Out Alerts:")
print(f"   - Critical Stock-Out Risks: {len(critical_stockouts)}")
print(f"   - Products at Warning Level: {len(stockout_df[stockout_df['risk_category'] == 'WARNING'])}")

print(f"\n📦 Contingency Options Available: {len(contingency_df)}")
print(f"   - Recommended Actions: {len(recommendations_df)}")

print("\n" + "="*80)
print("RECOMMENDED ACTIONS:")
print("="*80)

if len(critical_nodes) > 0:
    print("\n🔴 IMMEDIATE ACTIONS REQUIRED:")
    for _, node in critical_nodes.iterrows():
        print(f"   - {node['node_name']}: Risk Score {node['composite_risk']:.0f}")
        node_recs = recommendations_df[recommendations_df['vulnerable_item'].str.contains(node['node_name'][:5], na=False)]
        if len(node_recs) > 0:
            for _, rec in node_recs.head(2).iterrows():
                print(f"     → Procure {rec['contingency_item']} from {rec['vendor']} (${rec['cost_estimate']:,})")

if len(warning_nodes) > 0:
    print("\n🟡 MONITOR CLOSELY:")
    for _, node in warning_nodes.iterrows():
        print(f"   - {node['node_name']}: Risk Score {node['composite_risk']:.0f}")

print("\n" + "="*80)
print("HACKATHON VALUE PROPOSITION:")
print("="*80)
print("""
This prototype demonstrates how DHA data fused with operational data can:

1. 📈 Move Marine Corps blood support from REACTIVE resupply to PREDICTIVE sustainment
2. 🎯 Give decision-makers practical tools to preserve survivability when a central hub is constrained
3. 🗺️ Visualize risk across hub-and-spoke network in real-time
4. 🔔 Provide early warning of stock-out conditions before mission impact becomes acute
5. 💡 Evaluate market-aware contingency sourcing options when hub becomes constrained

The tool shows that the real readiness problem is not simply stock on hand, but whether 
viable blood support can move from hub to spoke in time to sustain distributed maritime operations.
""")

print("✅ Blood Logistics Decision Support Tool Prototype Complete!")